In [18]:
import re
from collections import Counter

class SebTokenizer:
    def __init__(self, vocab_size=4096):
        self.vocab_size = vocab_size
        self.token_to_id = {}
        self.id_to_token = {}

    def train(self, text):
        words = re.findall(r"\S+|\s+|[^\w\s]", text, re.UNICODE)

        vocab = Counter()

        for word in words:
            symbols = list(word) + ["</w>"]
            vocab[tuple(symbols)] += 1

        chars = sorted(set(
            symbol
            for word in vocab
            for symbol in word
        ))

        tokens = list(chars)

        while len(tokens) < self.vocab_size:
            pairs = Counter()

            for word, frequency in vocab.items():
                for i in range(len(word) - 1):
                    pairs[(word[i], word[i + 1])] += frequency

            if not pairs:
                break

            best_pair, count = pairs.most_common(1)[0]

            if count < 2:
                break

            new_token = best_pair[0] + best_pair[1]

            if new_token in tokens:
                break

            tokens.append(new_token)

            new_vocab = Counter()

            for word, frequency in vocab.items():
                new_word = []
                i = 0

                while i < len(word):
                    if (
                        i < len(word) - 1
                        and (word[i], word[i + 1]) == best_pair
                    ):
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1

                new_vocab[tuple(new_word)] += frequency

            vocab = new_vocab

        special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

        tokens = special_tokens + [
            token for token in tokens
            if token not in special_tokens
        ]

        tokens = tokens[:self.vocab_size]

        self.token_to_id = {
            token: i for i, token in enumerate(tokens)
        }

        self.id_to_token = {
            i: token for token, i in self.token_to_id.items()
        }

    def encode(self, text):
        tokens = []

        for word in re.findall(r"\S+|\s+|[^\w\s]", text, re.UNICODE):
            symbols = list(word) + ["</w>"]

            while len(symbols) > 1:
                possible_pairs = [
                    (symbols[i], symbols[i + 1])
                    for i in range(len(symbols) - 1)
                ]

                valid_pairs = [
                    pair
                    for pair in possible_pairs
                    if pair[0] + pair[1] in self.token_to_id
                ]

                if not valid_pairs:
                    break

                pair = max(
                    valid_pairs,
                    key=lambda x: len(x[0] + x[1])
                )

                merged = pair[0] + pair[1]

                new_symbols = []
                i = 0

                while i < len(symbols):
                    if (
                        i < len(symbols) - 1
                        and (symbols[i], symbols[i + 1]) == pair
                    ):
                        new_symbols.append(merged)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1

                symbols = new_symbols

            for symbol in symbols:
                if symbol == "</w>":
                    continue

                if symbol in self.token_to_id:
                    tokens.append(self.token_to_id[symbol])
                else:
                    tokens.append(self.token_to_id["<UNK>"])

        return tokens

    def decode(self, ids):
        output = ""

        for token_id in ids:
            token = self.id_to_token.get(token_id, "<UNK>")

            if token == "</w>":
                continue

            if token not in ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]:
                output += token

        return output

In [19]:
tokenizer = SebTokenizer(vocab_size=4096)

tokenizer.train(text)

print("Vocabulary:", len(tokenizer.token_to_id))

Vocabulary: 231


In [20]:
sample = "SebAI is a language model built from scratch."

encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)

print("Original:")
print(sample)

print("\nToken IDs:")
print(encoded)

print("\nDecoded:")
print(decoded)

Original:
SebAI is a language model built from scratch.

Token IDs:
[55, 38, 45, 38, 105, 38, 81, 38, 68, 38, 106, 38, 70, 38, 111]

Decoded:
SebAI</w> </w>is</w> </w>a</w> </w>language</w> </w>model</w> </w>built</w> </w>from</w> </w>scratch.</w>


In [21]:
for token_id in encoded:
    print(token_id, repr(tokenizer.id_to_token[token_id]))

55 'SebAI</w>'
38 ' </w>'
45 'is</w>'
38 ' </w>'
105 'a</w>'
38 ' </w>'
81 'language</w>'
38 ' </w>'
68 'model</w>'
38 ' </w>'
106 'built</w>'
38 ' </w>'
70 'from</w>'
38 ' </w>'
111 'scratch.</w>'
